# 01 · Whisper 语音识别：从转写到 WER 评测

**硬件**：🟢 CPU 可跑（默认用 whisper-small ~460MB；有 GPU 自动切换 large-v3-turbo）

## 本 notebook 你将学到

1. Whisper 的 encoder-decoder 架构：梅尔频谱进，文本 token 出
2. 转写、时间戳两种任务如何通过 **特殊 token 前缀** 切换（Whisper 的精髓设计）
3. 用 WER（词错误率）科学评测，而不是"听着挺准"
4. faster-whisper（CTranslate2 int8）带来的数倍加速

对应理论：[theory.md](../theory.md) 第 2 节。

In [ ]:
%pip install -q torch transformers datasets soundfile jiwer faster-whisper accelerate

In [ ]:
import torch

if torch.cuda.is_available():
    device, model_id = "cuda:0", "openai/whisper-large-v3-turbo"
else:
    device, model_id = "cpu", "openai/whisper-small"
print(f"device={device}, model={model_id}")

## 1. 准备测试音频

用 LibriSpeech 的样本（自带人工转写的参考文本，方便后面算 WER）。你也可以把 `wav` 换成自己的录音。

In [ ]:
from datasets import load_dataset
from IPython.display import Audio as AudioPlayer, display
import soundfile as sf

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample = ds[0]
wav, sr = sample["audio"]["array"], sample["audio"]["sampling_rate"]
reference = sample["text"].lower()

sf.write("sample.wav", wav, sr)  # 存成文件，后面 faster-whisper 用
print(f"时长 {len(wav)/sr:.1f}s | 参考转写: {reference}")
display(AudioPlayer(wav, rate=sr))

## 2. Whisper 转写

Whisper 把任务编码成 decoder 的**前缀 token**：

```
<|startoftranscript|> <|en|> <|transcribe|> <|notimestamps|> 正文...
```

换语言 = 换 `<|en|>`，换任务（翻译成英文）= 把 `<|transcribe|>` 换成 `<|translate|>`。一个模型，多个任务，全靠 prompt——这是 2022 年就有的"提示工程"。

In [ ]:
from transformers import pipeline
import time

asr = pipeline("automatic-speech-recognition", model=model_id, device=device)

t0 = time.perf_counter()
result = asr({"array": wav, "sampling_rate": sr})
t_hf = time.perf_counter() - t0

print(f"[{t_hf:.2f}s] {result['text']}")

In [ ]:
# 带时间戳（字幕场景）：Whisper 预测 <|0.00|> 这样的时间 token
result_ts = asr({"array": wav, "sampling_rate": sr}, return_timestamps=True)
for chunk in result_ts["chunks"]:
    print(f"{chunk['timestamp']}  {chunk['text']}")

## 3. 科学评测：WER

WER = (替换 + 删除 + 插入) / 参考词数。注意评测前要做**文本归一化**（大小写、标点），否则 `Hello.` vs `hello` 会被算成错误。

In [ ]:
import jiwer

norm = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
])

# 在整个 dummy 数据集上算平均 WER
refs, hyps = [], []
for s in ds:
    refs.append(norm(s["text"]))
    hyps.append(norm(asr({"array": s["audio"]["array"], "sampling_rate": sr})["text"]))

wer = jiwer.wer(refs, hyps)
print(f"WER on {len(ds)} samples: {wer:.2%}")
print("（LibriSpeech clean 朗读音频很简单；真实场景的口音/噪声/专有名词才是难点）")

## 4. faster-whisper：生产级加速

同一个模型权重，用 CTranslate2 重写推理 + int8 量化，CPU 上通常快 3–5 倍。这是"研究模型 → 生产部署"的经典案例。

In [ ]:
from faster_whisper import WhisperModel

fw_size = "large-v3-turbo" if torch.cuda.is_available() else "small"
fw = WhisperModel(fw_size, device="cuda" if torch.cuda.is_available() else "cpu",
                  compute_type="float16" if torch.cuda.is_available() else "int8")

t0 = time.perf_counter()
segments, info = fw.transcribe("sample.wav")
text_fw = " ".join(s.text for s in segments)
t_fw = time.perf_counter() - t0

print(f"transformers: {t_hf:.2f}s")
print(f"faster-whisper({fw.model.compute_type if hasattr(fw, 'model') else 'int8'}): {t_fw:.2f}s")
print(f"检测语言: {info.language} (p={info.language_probability:.2f})")
print(f"转写: {text_fw}")

## 练习

1. 录一段自己的中文语音（或找一段中文播客），测试 `language="zh"` 的转写质量；再试 `task="translate"` 直接出英文。
2. 给音频加高斯噪声（SNR 10dB），观察 WER 恶化多少——体会"鲁棒性来自数据"的含义。
3. 对比 `whisper-small` 和 `whisper-large-v3-turbo` 的 WER/速度曲线，思考生产中怎么选。
4. 进阶：试试 NVIDIA Parakeet TDT（见 [landscape.md](../landscape.md)），对比流式场景的延迟差异。

**下一站**：[02_kokoro_tts.ipynb](02_kokoro_tts.ipynb) — 反方向：文本变语音。